# Line Following - Live Demo

In this notebook, we'll use classical computer vision (OpenCV) instead of a trained neural network to make JetBot follow a strip of tape on the floor. This is the same idea as the `jetbot_line_follower.py` script, adapted into a notebook so you can tune everything live with sliders, just like in the road following demo.

### How it works

1. Grab a frame from the camera
2. Look only at a strip near the bottom of the frame (closest to the robot)
3. Threshold that strip to find the tape (dark tape on light floor, light tape on dark floor, or a specific tape color)
4. Find the horizontal center of the tape blob
5. Steer with a PD controller to keep that center in the middle of the frame

No model file is needed for this notebook -- it's pure image processing.

### Camera, Robot, and Imports

In [ ]:
import time
import cv2
import numpy as np
import ipywidgets
import traitlets
from IPython.display import display
from jetbot import Robot, Camera, bgr8_to_jpeg

robot = Robot()
camera = Camera()

image_widget = ipywidgets.Image()
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)


We'll also add a small widget that shows the black-and-white **mask** the robot is using to see the tape. This is very useful for tuning the threshold -- the tape should show up as a solid white blob and everything else should be black.

In [ ]:
mask_widget = ipywidgets.Image()

display(ipywidgets.HBox([image_widget, mask_widget]))


### Detection Settings

Pick how the tape should be detected, and tune the threshold live.

In [ ]:
# "dark"  -> dark tape on a lighter floor (e.g. black tape on wood/tile)
# "light" -> light tape on a darker floor (e.g. white tape on dark floor)
# "color" -> a specific colored tape (e.g. red, blue, green electrical tape)
detection_mode_dropdown = ipywidgets.Dropdown(
    options=['dark', 'light', 'color'],
    value='light',
    description='mode'
)

brightness_threshold_slider = ipywidgets.IntSlider(min=0, max=255, step=1, value=160, description='threshold')

# Only used when detection_mode is "color". Defaults below are tuned for a fairly
# saturated RED tape -- adjust to match your tape color.
color_lower_hsv = np.array([0, 120, 70])
color_upper_hsv = np.array([10, 255, 255])

# Only look at the bottom portion of the frame (closer to the robot).
# 0.0 = top of image, 1.0 = bottom of image.
roi_top_fraction_slider = ipywidgets.FloatSlider(min=0.0, max=0.9, step=0.05, value=0.6, description='ROI top')

display(detection_mode_dropdown, brightness_threshold_slider, roi_top_fraction_slider)


### Tape Detection Function

Given a camera frame, this returns the normalized horizontal offset of the tape's center from the middle of the image, in the range `[-1.0, 1.0]`. It returns `None` if no tape is found.

In [ ]:
def find_tape_center(frame):
    h, w = frame.shape[:2]
    roi_top = int(h * roi_top_fraction_slider.value)
    roi = frame[roi_top:h, :]

    if detection_mode_dropdown.value == 'color':
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, color_lower_hsv, color_upper_hsv)
    else:
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        if detection_mode_dropdown.value == 'dark':
            # tape is darker than floor -> threshold below the value
            _, mask = cv2.threshold(gray, brightness_threshold_slider.value, 255, cv2.THRESH_BINARY_INV)
        else:  # "light"
            _, mask = cv2.threshold(gray, brightness_threshold_slider.value, 255, cv2.THRESH_BINARY)

    # Clean up noise
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)

    # Update the mask preview -- pad it back to full frame height so it's easy to compare
    full_mask = np.zeros((h, w), dtype=np.uint8)
    full_mask[roi_top:h, :] = mask
    mask_widget.value = bgr8_to_jpeg(cv2.cvtColor(full_mask, cv2.COLOR_GRAY2BGR))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    # Pick the largest contour (assumed to be the tape)
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 50:  # too small, likely noise
        return None

    M = cv2.moments(largest)
    if M['m00'] == 0:
        return None

    cx = int(M['m10'] / M['m00'])
    # Normalize: -1.0 (far left) to +1.0 (far right), 0.0 = centered
    offset = (cx - (w / 2)) / (w / 2)
    return offset


### Driving Sliders

Just like the road following demo:

1. **speed gain**: start your JetBot moving by increasing this
2. **steering gain**: if JetBot wobbles, reduce this until it's smooth
3. **steering kd**: derivative gain, helps damp oscillation
4. **steering bias**: if JetBot drifts to one side of the tape even when centered, adjust this

> Tune with a lower speed first to get smooth line-following behavior.

In [ ]:
speed_gain_slider = ipywidgets.FloatSlider(min=0.0, max=0.3, step=0.001, value=0.13, description='speed gain')
steering_gain_slider = ipywidgets.FloatSlider(min=0.0, max=0.1, step=0.001, value=0.02, description='steering gain')
steering_dgain_slider = ipywidgets.FloatSlider(min=0.0, max=0.1, step=0.001, value=0.03, description='steering kd')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.1, max=0.1, step=0.001, value=0.0, description='steering bias')

display(speed_gain_slider, steering_gain_slider, steering_dgain_slider, steering_bias_slider)


In [ ]:
offset_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='offset')
steering_slider = ipywidgets.FloatSlider(min=-1.0, max=1.0, description='steering')
speed_slider = ipywidgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='speed')
tape_found_widget = ipywidgets.Valid(value=False, description='tape found')

display(ipywidgets.HBox([speed_slider, ipywidgets.VBox([offset_slider, steering_slider, tape_found_widget])]))


### Control Loop

This function is called on every new camera frame. It:

1. Runs `find_tape_center` to locate the tape
2. Computes a PD steering correction from the offset
3. Drives the motors -- and if the tape is briefly lost, keeps going with the last known heading, then spins slowly to search for it if it stays lost too long

In [ ]:
# How long to keep going with the last known direction after losing the tape (seconds)
LOST_TAPE_COAST_TIME = 0.5
# Speed used while spinning to search for the tape once it's been lost too long
SEARCH_SPEED = 0.18

angle = 0.0
angle_last = 0.0
last_seen_offset = 0.0
last_seen_time = time.time()

def execute(change):
    global angle, angle_last, last_seen_offset, last_seen_time

    image = change['new']
    offset = find_tape_center(image)
    now = time.time()

    speed_slider.value = speed_gain_slider.value

    if offset is not None:
        tape_found_widget.value = True
        offset_slider.value = offset
        last_seen_offset = offset
        last_seen_time = now

        angle = offset
        pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
        angle_last = angle

        steering_slider.value = pid + steering_bias_slider.value

        robot.left_motor.value = max(min(speed_slider.value + steering_slider.value, 1.0), 0.0)
        robot.right_motor.value = max(min(speed_slider.value - steering_slider.value, 1.0), 0.0)
    else:
        tape_found_widget.value = False
        time_since_seen = now - last_seen_time

        if time_since_seen < LOST_TAPE_COAST_TIME:
            # keep going with the last motor command -- do nothing this frame
            pass
        else:
            # spin slowly toward the side where the tape was last seen
            if last_seen_offset >= 0:
                robot.set_motors(SEARCH_SPEED, -SEARCH_SPEED)
            else:
                robot.set_motors(-SEARCH_SPEED, SEARCH_SPEED)

execute({'new': camera.value})


>**WARNING**: This code will move the robot!! Please make sure your robot has clearance. Place it near (but not on) your tape/track before running the next cell so you can see the mask and offset sliders react before it starts driving.

Now attach the `execute` function to the camera so it runs on every new frame:

In [ ]:
camera.observe(execute, names='value')


Your JetBot should now be generating new motor commands with each camera frame. Place it on your tape and see whether it follows the line.

If the mask widget above doesn't show a clean white blob on the tape, stop the robot (next cell), then re-tune `detection mode`, `threshold`, and `ROI top` before trying again.

To stop this behavior, run the cell below to unattach the callback.

In [ ]:
camera.unobserve(execute, names='value')

time.sleep(0.1)  # let any in-flight frame finish processing

robot.stop()


Close the camera connection properly so it can be used in other notebooks.

In [ ]:
camera.stop()


In [ ]:
robot.stop()


### Conclusion

That's it for the line following live demo! Unlike the road following notebook, there's no model to train -- all the tuning happens through the `mode`, `threshold`, `ROI top`, and PD gain sliders above. If JetBot loses the tape at corners, try lowering `speed gain` or widening the ROI so it sees more of the tape ahead.